<a href="https://colab.research.google.com/github/JariusF10-8/bt4222/blob/main/notebooks/features/feature_engineering_chris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIND-small: category encoding, historical popularity/CTR and entity prominence

This notebook builds three feature groups from the canonical preprocessing output:

| Part | Interim / insight item | Output |
|---|---|---|
| A | Section 5.1 `Category encoding` | Integer codes for category and (category, subcategory), fitted on training articles, plus a history subcategory match feature |
| B | Section 5.1 `Historical popularity / CTR` | Article, subcategory, user and user-article click signals computed only from interactions strictly before each impression |
| C | Insight 4 `Entity prominence` | Title-start entity indicators, normalised earliest positions, and position-weighted TransE article vectors |

Reusable functions live in `src/category_encoding.py`, `src/history_features.py`, `src/entity_prominence.py` and `src/feature_diagnostics.py`, with unit tests in `tests/`. Validation labels are only used to check features; test labels are never used for evaluation here.

**Run:** top to bottom. Colab needs about 6 GB RAM and 10 to 15 minutes. Dependencies are pandas and NumPy only.

## 1. Setup

In Colab, the cell asks for the repository ZIP when the code is not already available (same approach as `feature_engineering_zp.ipynb`). Locally, it finds the repository from the working directory.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules


def find_project_root():
    configured = os.environ.get('BT4222_PROJECT_ROOT')
    options = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    for folder in options:
        if (folder / 'src/data_loading.py').is_file():
            return folder.resolve()
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None and IN_COLAB:
    import io
    import tempfile
    import zipfile
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
    target = Path(tempfile.mkdtemp(prefix='bt4222_', dir='/content'))
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(target)
    PROJECT_ROOT = next(target.rglob('src/data_loading.py')).parent.parent
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(PROJECT_ROOT / 'requirements.txt')], check=True)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook inside the repository or set BT4222_PROJECT_ROOT.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ['BT4222_PROJECT_ROOT'] = str(PROJECT_ROOT)
print('Repository found:', PROJECT_ROOT.name)

Repository found: bt4222-main


In [2]:
import gc
import hashlib
import json
import time

import numpy as np
import pandas as pd
from IPython.display import display

from src.preprocessing import build_dataset
from src.category_encoding import (RARE, UNKNOWN, candidate_history_subcategory_match,
                                   encode_categories, fit_category_vocab, subcategory_key)
from src.history_features import (FEATURE_COLUMNS as HISTORY_COLUMNS, SMOOTHING, WINDOW_HOURS,
                                  build_exposure_log, compute_history_features)
from src.entity_prominence import (aggregate_entity_vectors, article_prominence_features,
                                   entity_mentions, load_entity_vectors, mention_weights)
from src.feature_diagnostics import impression_metrics, paired_difference, univariate_table

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SHARED = Path('/content/drive/MyDrive/BT4222 Group Project')
    zipeng_v2 = SHARED / 'feature_engineering_zp/data/processed/v2'
    default_processed = zipeng_v2 if (zipeng_v2 / 'manifest.json').is_file() else SHARED / 'feature_engineering_chris/data/processed/v1'
    os.environ.setdefault('BT4222_PROCESSED_DIR', str(default_processed))
    os.environ.setdefault('BT4222_CHRIS_FEATURES_DIR', str(SHARED / 'feature_engineering_chris/outputs/features/chris_v1'))

RAW_DIR = Path(os.environ.get('MIND_RAW_DIR', PROJECT_ROOT / 'data/raw'))
PROCESSED_DIR = Path(os.environ.get('BT4222_PROCESSED_DIR', PROJECT_ROOT / 'data/processed/v1'))
OUTPUT_DIR = Path(os.environ.get('BT4222_CHRIS_FEATURES_DIR', PROJECT_ROOT / 'outputs/features/chris_v1'))

if not (PROCESSED_DIR / 'manifest.json').is_file():
    display(build_dataset(RAW_DIR, PROCESSED_DIR, require_test=True))
manifest = json.loads((PROCESSED_DIR / 'manifest.json').read_text())
PARTITIONS = manifest['partitions']
if PARTITIONS != ['train', 'validation', 'test']:
    raise ValueError(f'Expected train, validation and test partitions; found {PARTITIONS}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for partition in PARTITIONS:
    (OUTPUT_DIR / partition).mkdir(exist_ok=True)
print('Processed dataset version:', manifest['dataset_version'], '| partitions:', ', '.join(PARTITIONS))
display(pd.read_csv(PROCESSED_DIR / 'split_summary.csv')[['partition', 'earliest_time', 'latest_time', 'events', 'candidate_exposures', 'clicked_candidates']])

Processed dataset version: v1 | partitions: train, validation, test


,partition,earliest_time,latest_time,events,candidate_exposures,clicked_candidates
0,train,2019-11-09 00:00:19,2019-11-12 23:59:25,95071,3389342,141558
1,validation,2019-11-13 00:01:00,2019-11-14 23:59:13,61894,2454102,94786
2,test,2019-11-15 00:00:01,2019-11-15 23:58:03,73152,2740998,111383


## 2. Memory-light loaders

`load_partition` restores every column, including all text and parsed entity lists. The features here only need a few columns at a time, so these loaders read just those columns and store repeated IDs as categories. This keeps the full 8.6 million candidate log below about 3 GB.

In [3]:
def read_news(partition, columns):
    news = pd.read_csv(PROCESSED_DIR / partition / 'news.csv', usecols=columns, dtype='string', keep_default_na=False)
    if 'metadata_missing' in news:
        news['metadata_missing'] = news['metadata_missing'].eq('True')
    for column in ('title_entities_parsed', 'abstract_entities_parsed'):
        if column in news:
            news[column] = news[column].map(json.loads)
    return news


def read_events(partition, history=False):
    columns = ['event_key', 'user_id', 'timestamp'] + (['history_ids'] if history else [])
    events = pd.read_csv(PROCESSED_DIR / partition / 'events.csv', usecols=columns, keep_default_na=False,
                         dtype={'event_key': 'string', 'user_id': 'string'}, parse_dates=['timestamp'])
    if history:
        events['history_ids'] = events['history_ids'].map(json.loads)
    return events


def read_candidates(partition, label=True):
    columns = ['event_key', 'candidate_position', 'news_id'] + (['label'] if label else [])
    return pd.read_csv(PROCESSED_DIR / partition / 'candidates.csv', usecols=columns, keep_default_na=False,
                       dtype={'event_key': 'category', 'news_id': 'category', 'candidate_position': 'int32', 'label': 'int8'})


def write_candidate_table(frame, partition, name):
    path = OUTPUT_DIR / partition / f'{name}.csv.gz'
    frame.to_csv(path, index=False, float_format='%.6g', compression={'method': 'gzip', 'compresslevel': 5})
    return path


CATEGORY_COLUMNS = ['news_id', 'category', 'subcategory', 'metadata_missing']
diagnostics = {}
started = time.time()

## Part A. Category encoding

**What the interim report found.** 17 categories, 264 subcategory names, 13 subcategory names that sit under more than one category (for example `video`, `news`, `recipes`), and a long tail of rare subcategories. Category is never unseen in validation, while a few subcategories are.

**Encoding choices.**

1. `category_id`: sorted training categories get 1 to K and 0 means unknown. This is the same rule as `feature_engineering_zp.ipynb`, so both notebooks share one mapping.
2. `subcategory_id`: the (category, subcategory) **pair**, so `news/video` and `video/video` stay different. Pairs with fewer than `MIN_ARTICLES` training articles share one `__RARE__` bucket per category, so the bucket still carries its parent category. A pair never seen in training also maps to its category's rare bucket, and a category never seen in training maps to 0.
3. Everything is fitted on `train/news.csv` excluding `metadata_missing` rows, then applied unchanged to validation and test.

Integer codes feed embedding layers in the neural models (for example `nn.Embedding(n_subcategories, 16)`) and can be passed as categorical features to tree models. One-hot category columns already exist in Zipeng's notebook, so they are not duplicated here.

`MIN_ARTICLES = 5` follows the interim report's rare-subcategory statistic. It is a preprocessing choice, not a tuned value.

In [4]:
MIN_ARTICLES = 5
train_news_categories = read_news('train', CATEGORY_COLUMNS)
vocab = fit_category_vocab(train_news_categories, min_articles=MIN_ARTICLES)
pairs = vocab['pairs']
print(f"Categories (excluding unknown): {len(vocab['category']) - 1}")
print(f"Training (category, subcategory) pairs: {len(pairs)}; rare pairs (< {MIN_ARTICLES} articles): {int(pairs['bucket'].eq(RARE).sum())}, "
      f"covering {int(pairs.loc[pairs['bucket'].eq(RARE), 'train_articles'].sum())} articles")
print(f"Subcategory codes after merging (including 0 = unknown): {len(vocab['subcategory'])}")
names = pairs.groupby('subcategory')['category'].nunique()
print('Subcategory names under more than one category:', ', '.join(sorted(names[names > 1].index)))
display(vocab['category'])
display(vocab['subcategory'].head(12))

Categories (excluding unknown): 17
Training (category, subcategory) pairs: 276; rare pairs (< 5 articles): 107, covering 192 articles
Subcategory codes after merging (including 0 = unknown): 187
Subcategory names under more than one category: animals, causes, holidays, humor, news, newstrends, personalfinance, pregnancyparenting, recipes, science, topnews, video, voices


,category_id,category,train_articles
0,0,__UNKNOWN__,0
1,1,autos,1425
2,2,entertainment,529
3,3,finance,2585
4,4,foodanddrink,2166
5,5,health,1673
6,6,kids,14
7,7,lifestyle,2092
8,8,middleeast,2
9,9,movies,561


,subcategory_id,category,bucket,train_articles,merged_subcategories
0,0,__UNKNOWN__,__UNKNOWN__,0,0
1,1,autos,__RARE__,13,7
2,2,autos,autosbuying,28,1
3,3,autos,autosclassics,108,1
4,4,autos,autosenthusiasts,216,1
5,5,autos,autoshybrids,6,1
6,6,autos,autosluxury,19,1
7,7,autos,autosmotorcycles,102,1
8,8,autos,autosnews,693,1
9,9,autos,autosownership,25,1


In [5]:
encoded_by_partition, coverage_rows = {}, []
for partition in PARTITIONS:
    news = train_news_categories if partition == 'train' else read_news(partition, CATEGORY_COLUMNS)
    encoded = encode_categories(news, vocab)
    encoded_by_partition[partition] = encoded
    exposures = read_candidates(partition, label=False)[['news_id']].astype(str).merge(
        encoded, on='news_id', how='left', validate='many_to_one')
    unseen_pairs = set(zip(news['category'], news['subcategory'])) - set(zip(pairs['category'], pairs['subcategory']))
    unseen_categories = sorted(set(news.loc[encoded['category_unknown'].eq(1).to_numpy(), 'category']) - {UNKNOWN})
    coverage_rows.append({
        'partition': partition, 'articles': len(encoded),
        'unknown_category_articles': int(encoded['category_unknown'].sum()),
        'unseen_categories': ', '.join(unseen_categories) or '-',
        'pairs_not_in_training': len(unseen_pairs),
        'backoff_articles': int(encoded['subcategory_backoff'].sum()),
        'exposures_unknown_category_%': round(100 * exposures['category_unknown'].mean(), 3),
        'exposures_subcategory_backoff_%': round(100 * exposures['subcategory_backoff'].mean(), 3),
    })
category_coverage = pd.DataFrame(coverage_rows)
display(category_coverage)

,partition,articles,unknown_category_articles,unseen_categories,pairs_not_in_training,backoff_articles,exposures_unknown_category_%,exposures_subcategory_backoff_%
0,train,43370,0,-,0,192,0.0,1.552
1,validation,38421,0,-,7,190,0.0,1.174
2,test,42416,1,games,14,212,0.0,1.867


### A2. History subcategory match

The encoding becomes useful for ranking once it is compared with the user's supplied history. `candidate_history_subcategory_share` is the share of the impression's supplied history (known subcategories only, repeats kept) that falls in the candidate's (category, subcategory) pair. It uses raw pairs, so it needs no fitted vocabulary and no labels. Unknown subcategories never count as a match, and an empty history gives share 0 with `history_subcategory_missing = 1`. It complements Zipeng's top-level `candidate_history_category_share`.

In [6]:
subcategory_keys = {}
for partition in PARTITIONS:
    news = train_news_categories if partition == 'train' else read_news(partition, CATEGORY_COLUMNS)
    subcategory_keys[partition] = pd.Series(subcategory_key(news).to_numpy(), index=news['news_id'].astype(str).to_numpy())
    events = read_events(partition, history=True)
    candidates = read_candidates(partition)
    match = candidate_history_subcategory_match(events, candidates, news)
    columns = ['event_key', 'candidate_position', 'candidate_history_subcategory_share',
               'candidate_history_subcategory_count', 'history_subcategory_missing']
    write_candidate_table(match[columns], partition, 'candidate_subcategory_match')
    if partition == 'validation':
        check = match.assign(label=candidates['label'].to_numpy())
        diagnostics['subcategory_match_validation'] = univariate_table(check, ['candidate_history_subcategory_share'])
    print(f'{partition}: {len(match):,} candidate rows; mean share {match["candidate_history_subcategory_share"].mean():.4f}; '
          f'empty-history rows {int(match["history_subcategory_missing"].sum()):,}')
    del events, candidates, match
    _ = gc.collect()
display(diagnostics['subcategory_match_validation'])

train: 3,389,342 candidate rows; mean share 0.0408; empty-history rows 61,834


validation: 2,454,102 candidate rows; mean share 0.0357; empty-history rows 58,608


test: 2,740,998 candidate rows; mean share 0.0324; empty-history rows 82,907


,feature,direction,impressions,AUC,MRR,nDCG@5,nDCG@10
0,candidate_history_subcategory_share,higher is better,61894,0.5885,0.2729,0.2894,0.3478


**Reading the check.** Each validation impression is ranked by this single feature and compared with the clicks (per-impression AUC, MRR and nDCG, the MIND definitions; a random ranking gives AUC 0.500 and nDCG@10 about 0.270). On its own this feature reaches validation AUC 0.589 and nDCG@10 0.348, clearly above random, which supports giving the encoded subcategory and the match feature to the scoring MLP.

## Part B. Historical popularity and CTR

**Rule.** A feature for an impression at time *t* only uses exposures and clicks with time strictly less than *t*. The impression itself and anything later never enter, and no current-event click totals are used.

**Two modes, same column names.**

| Mode | Training rows | Validation and test rows |
|---|---|---|
| `online` | interactions before *t* | interactions before *t*, including earlier impressions in the same period (simulates a platform that logs clicks as they happen) |
| `frozen` | interactions before *t* | only interactions before the partition starts (the interim report's "freeze training-derived aggregates" rule; test sees train and validation) |

`online` is the realistic setting for news, where articles live for hours. `frozen` matches the interim wording but creates a train and serve mismatch: training rows always have fresh counts, while about half of validation exposures are articles never seen before the partition. Both are exported so the models can run this ablation. Online test features read labels of earlier test impressions as inputs; they never use the impression being scored. Confirm this protocol with Dr Wang at the consultation before reporting online test results.

**Features** (all computed per candidate row):

| Column | Meaning |
|---|---|
| `art_prior_exposures`, `art_prior_clicks` | Times the article was shown / clicked before *t* |
| `art_exposures_{1,6,24}h`, `art_clicks_{1,6,24}h` | Same, inside the last 1, 6 and 24 hours |
| `art_prior_ctr`, `art_ctr_24h` | Smoothed click rate, shrunk towards the article's subcategory rate |
| `subcat_prior_ctr` | Smoothed click rate of the (category, subcategory) pair, shrunk towards the training click rate |
| `art_hours_since_first_exposure`, `art_seen_before` | Article age proxy (MIND has no publish time) and whether it was ever shown before *t* |
| `user_prior_events`, `user_prior_exposures`, `user_prior_clicks`, `user_prior_ctr` | The user's own activity before *t* |
| `user_article_prior_exposures`, `user_article_prior_clicks`, `user_article_skipped_before` | Whether this user was already shown this article and did not click it |

Smoothing: `rate = (clicks + m x prior) / (exposures + m)` with m = 100 for subcategories, 20 for articles and users. The global prior is the training click rate, one constant. Smoothing lets a new article start at its topic's rate instead of 0 or 1. These values are reasonable defaults, not tuned.

**Fix compared with the previous project.** Group 10 computed CTR on the negative-sampled training table, so every article sat near 1/5, the row's own click was inside its CTR, and unseen test articles all received the same constant. `src/sampling.py` already warns about this; here every rate uses the full candidate log and only earlier rows.

In [7]:
log_inputs = {partition: (read_events(partition), read_candidates(partition)) for partition in PARTITIONS}
log = build_exposure_log(log_inputs, subcategory_keys)
candidate_keys = {partition: candidates[['event_key', 'candidate_position']].copy() for partition, (_, candidates) in log_inputs.items()}
del log_inputs
_ = gc.collect()
print(f'Exposure log: {len(log):,} candidate rows, {log["event"].nunique():,} impressions, '
      f'{log["article"].nunique():,} articles, {log["user"].nunique():,} users')
print('Training click rate (global prior):', round(float(log.loc[log['partition'].eq(0), 'label'].mean()), 5))

Exposure log: 8,584,442 candidate rows, 230,117 impressions, 22,771 articles, 94,057 users
Training click rate (global prior): 0.04177


In [8]:
history_frames = {}
for mode in ('online', 'frozen'):
    tick = time.time()
    history_frames[mode] = compute_history_features(log, mode=mode)
    print(f'{mode}: computed {len(HISTORY_COLUMNS)} features in {time.time() - tick:.0f}s')

online: computed 20 features in 32s


frozen: computed 20 features in 28s


### B2. Leakage audit on the real data

Two checks beyond the unit tests:

1. **Brute force.** For random validation and test rows, recount the article, user and user-article history directly by filtering the log, and compare with the fast implementation.
2. **Perturbation.** Flip every label at or after a cut time inside the validation period and recompute. Features of rows at or before the cut must not change, which shows that current and future labels never reach the features.

In [9]:
def brute_force(row, mode):
    t = log.at[row, 't']
    upper = t
    if mode == 'frozen' and log.at[row, 'partition'] != 0:
        upper = min(t, log.loc[log['partition'].eq(log.at[row, 'partition']), 't'].min())
    past = log.loc[log['t'].lt(upper)]
    same = past.loc[past['article'].eq(log.at[row, 'article'])]
    day = same.loc[same['t'].ge(t - 24 * 3600)]
    pair = same.loc[same['user'].eq(log.at[row, 'user'])]
    return {'art_prior_exposures': len(same), 'art_prior_clicks': int(same['label'].sum()),
            'art_exposures_24h': len(day), 'art_clicks_24h': int(day['label'].sum()),
            'user_article_prior_exposures': len(pair), 'user_article_prior_clicks': int(pair['label'].sum())}


rng = np.random.default_rng(42)
mismatches, checked = 0, 0
for mode in ('online', 'frozen'):
    for code_ in (1, 2):
        rows = rng.choice(np.flatnonzero(log['partition'].to_numpy() == code_), size=40, replace=False)
        for row in rows:
            expected = brute_force(row, mode)
            got = history_frames[mode].loc[row, list(expected)]
            mismatches += int((got.to_numpy() != np.array(list(expected.values()))).any())
            checked += 1
print(f'Brute-force check: {checked} rows, {mismatches} mismatches')
assert mismatches == 0

validation_t = log.loc[log['partition'].eq(1), 't']
cut = int(validation_t.median())
perturbed = log.copy()
later = perturbed['t'].ge(cut).to_numpy()
perturbed.loc[later, 'label'] = 1 - perturbed.loc[later, 'label']
p0 = float(log.loc[log['partition'].eq(0), 'label'].mean())
reference = compute_history_features(log, mode='online', prior_ctr=p0)
changed = compute_history_features(perturbed, mode='online', prior_ctr=p0)
keep = log['t'].le(cut).to_numpy()
unchanged = bool((reference.loc[keep].to_numpy() == changed.loc[keep].to_numpy()).all())
moved = int((reference.loc[~keep].to_numpy() != changed.loc[~keep].to_numpy()).any(axis=1).sum())
print(f'Perturbation check: {int(later.sum()):,} labels flipped; rows at or before the cut unchanged: {unchanged}; '
      f'later rows that changed: {moved:,}')
assert unchanged
del perturbed, reference, changed
_ = gc.collect()

Brute-force check: 160 rows, 0 mismatches


Perturbation check: 3,968,056 labels flipped; rows at or before the cut unchanged: True; later rows that changed: 3,967,751


### B3. How informative is each feature on validation?

Each validation impression is ranked by one feature at a time. Features where fewer views mean more clicks are flipped (marked "lower is better"). This is a screening check, not a model: the final model combines the features and decides the weights.

In [10]:
validation_rows = np.flatnonzero(log['partition'].to_numpy() == 1)
lower_is_better = {'art_hours_since_first_exposure': -1, 'user_article_skipped_before': -1, 'user_article_prior_exposures': -1}
screen = []
for mode, frame in history_frames.items():
    scored = frame.iloc[validation_rows].reset_index(drop=True)
    scored['event_key'] = candidate_keys['validation']['event_key'].astype(str).to_numpy()
    scored['label'] = log['label'].to_numpy()[validation_rows]
    table = univariate_table(scored, HISTORY_COLUMNS, sign=lower_is_better)
    table.insert(0, 'mode', mode)
    screen.append(table)
history_screen = pd.concat(screen, ignore_index=True)
diagnostics['history_screen_validation'] = history_screen
wide = history_screen.pivot(index='feature', columns='mode', values=['AUC', 'nDCG@10']).reindex(HISTORY_COLUMNS)
display(wide.round(4))

seen = {mode: float(frame['art_seen_before'].to_numpy()[validation_rows].mean()) for mode, frame in history_frames.items()}
print('Validation exposures on articles already shown before (online vs frozen):',
      {k: f'{100 * v:.1f}%' for k, v in seen.items()})
print('Validation rows already shown to the same user and skipped (online):',
      f"{100 * history_frames['online']['user_article_skipped_before'].to_numpy()[validation_rows].mean():.1f}%")
skipped = history_frames['online']['user_article_skipped_before'].to_numpy()[validation_rows] == 1
labels = log['label'].to_numpy()[validation_rows]
print(f'Validation CTR when shown and skipped before: {100 * labels[skipped].mean():.2f}% vs not: {100 * labels[~skipped].mean():.2f}%')

AUC         nDCG@10        
mode                            frozen  online  frozen  online
feature                                                       
art_prior_exposures             0.4859  0.4724  0.2628  0.2584
art_prior_clicks                0.5015  0.5502  0.2776  0.3083
art_prior_ctr                   0.5637  0.6464  0.3109  0.3669
art_exposures_1h                0.5000  0.4682  0.2706  0.2623
art_clicks_1h                   0.5001  0.5622  0.2707  0.3192
art_exposures_6h                0.5039  0.4639  0.2744  0.2529
art_clicks_6h                   0.5048  0.5475  0.2757  0.3045
art_exposures_24h               0.5069  0.4718  0.2795  0.2591
art_clicks_24h                  0.5103  0.5516  0.2847  0.3079
art_ctr_24h                     0.5569  0.6464  0.3092  0.3674
art_hours_since_first_exposure  0.5172  0.5218  0.2769  0.2871
art_seen_before                 0.4868  0.4994  0.2606  0.2703
subcat_prior_ctr                0.5484  0.5620  0.2983  0.3023
user_prior_events               0.5000  0.5000  0.2705  0.2705
user_prior_exposures            0.5000  0.5000  0.2705  0.2705
user_prior_clicks               0.5000  0.5000  0.2705  0.2705
user_prior_ctr                  0.5000  0.5000  0.2705  0.2705
user_article_prior_exposures    0.5031  0.5374  0.2712  0.2951
user_article_prior_clicks       0.5009  0.5004  0.2719  0.2711
user_article_skipped_before     0.5034  0.5359  0.2713  0.2947

Validation exposures on articles already shown before (online vs frozen): {'online': '99.7%', 'frozen': '49.7%'}
Validation rows already shown to the same user and skipped (online): 15.4%
Validation CTR when shown and skipped before: 2.36% vs not: 4.14%


**Findings to carry into the report.**

- Online smoothed CTR (`art_prior_ctr`, `art_ctr_24h`) is the strongest single signal: validation AUC 0.646 and nDCG@10 0.367. Frozen CTR drops to AUC 0.564 and nDCG@10 0.311, because only 49.7% of validation exposures are on articles shown before the partition starts (99.7% online).
- Exposure counts alone rank **below** random (AUC 0.46 to 0.47): articles the platform pushes hardest are clicked less per view, so popularity must be measured as clicks per exposure, not views.
- User-level features score exactly 0.500. They are the same for every candidate in an impression, so on their own they cannot reorder candidates. They only help through interactions in the MLP (for example, letting users with little history rely more on popularity).
- `user_article_skipped_before` is a real negative signal from the impression logs: 15.4% of validation candidates were already shown to the same user and skipped, and their CTR is 2.36% against 4.14% for the rest.

In [11]:
mode_files = {}
for mode, frame in history_frames.items():
    for partition, code_ in zip(PARTITIONS, range(3)):
        rows = np.flatnonzero(log['partition'].to_numpy() == code_)
        out = pd.concat([candidate_keys[partition].reset_index(drop=True), frame.iloc[rows].reset_index(drop=True)], axis=1)
        if len(out) != len(candidate_keys[partition]):
            raise ValueError('Row count changed during export')
        mode_files[(mode, partition)] = write_candidate_table(out, partition, f'candidate_history_{mode}')
        print(f'{partition}: wrote candidate_history_{mode}.csv.gz ({len(out):,} rows)')
history_meta = {'smoothing': SMOOTHING, 'window_hours': list(WINDOW_HOURS),
                'global_prior_ctr': round(float(log.loc[log['partition'].eq(0), 'label'].mean()), 6)}
del history_frames, log
_ = gc.collect()

train: wrote candidate_history_online.csv.gz (3,389,342 rows)


validation: wrote candidate_history_online.csv.gz (2,454,102 rows)


test: wrote candidate_history_online.csv.gz (2,740,998 rows)


train: wrote candidate_history_frozen.csv.gz (3,389,342 rows)


validation: wrote candidate_history_frozen.csv.gz (2,454,102 rows)


test: wrote candidate_history_frozen.csv.gz (2,740,998 rows)


## Part C. Entity prominence (insight 4)

**Data check first.** Each entity annotation has `OccurrenceOffsets` (character positions) and `SurfaceForms`. The next cell confirms that `text[offset : offset + len(surface)]` equals the surface form in the raw title or abstract, so offsets can be trusted as positions.

**Features per article.**

| Column | Meaning |
|---|---|
| `has_entity_at_title_start` | An entity starts the title, allowing only leading punctuation (for example `'Trump' visits...`) |
| `title_start_entity_type`, `title_start_type_group` | Type code of that lead entity; group = person (P), organisation (O), place (G), other, or none |
| `title_entity_first_position` | Earliest title entity offset divided by title length in characters; 1.0 with `title_entity_absent = 1` when there is none |
| `abstract_entity_first_position` | Same for the abstract, kept separate as the guidance recommends |

**Position-weighted aggregation.** Each mention gets `w = section weight x exp(-decay x normalised position)`, title weight 1 and abstract weight `a`. An entity named in both title and abstract counts once with its larger weight, and the article vector is the weighted mean of TransE vectors. `a = 1, decay = 0` is the plain average, so the uniform baseline is one point of the same grid. Earlier position is treated as a hypothesis and tested below. Mentions with no offsets (a few hundred) are treated as the least prominent position.

In [12]:
ENTITY_COLUMNS = ['news_id', 'title', 'abstract', 'title_entities_parsed', 'abstract_entities_parsed']
alignment = []
for partition in PARTITIONS:
    news = read_news(partition, ENTITY_COLUMNS)
    for section in ('title', 'abstract'):
        matched = total = 0
        for text, entities in zip(news[section], news[f'{section}_entities_parsed']):
            for entity in entities:
                for surface, offset in zip(entity.get('SurfaceForms') or [], entity.get('OccurrenceOffsets') or []):
                    total += 1
                    matched += text[offset:offset + len(surface)] == surface
        alignment.append({'partition': partition, 'section': section, 'occurrences': total,
                          'offset_matches_surface_form_%': round(100 * matched / max(total, 1), 3)})
display(pd.DataFrame(alignment))

,partition,section,occurrences,offset_matches_surface_form_%
0,train,title,51121,100.0
1,train,abstract,92180,100.0
2,validation,title,44812,100.0
3,validation,abstract,79168,100.0
4,test,title,50263,100.0
5,test,abstract,88614,100.0


In [13]:
prominence_by_partition, mentions_by_partition, summary_rows = {}, {}, []
for partition in PARTITIONS:
    news = read_news(partition, ENTITY_COLUMNS)
    mentions = entity_mentions(news)
    features = article_prominence_features(news, mentions)
    mentions_by_partition[partition] = mentions
    prominence_by_partition[partition] = features
    title_first = mentions.loc[mentions['section'].eq('title') & mentions['first_offset'].ge(0)].groupby('news_id')['first_offset'].min()
    summary_rows.append({
        'partition': partition, 'articles': len(features), 'entity_mentions': len(mentions),
        'mentions_without_offsets': int(mentions['first_offset'].lt(0).sum()),
        'title_start_strict_offset_0': int(title_first.eq(0).sum()),
        'title_start_with_leading_punctuation': int(features['has_entity_at_title_start'].sum()),
        'title_entity_absent_%': round(100 * features['title_entity_absent'].mean(), 2),
        'abstract_entity_absent_%': round(100 * features['abstract_entity_absent'].mean(), 2),
        'median_title_first_position': round(float(features.loc[features['title_entity_absent'].eq(0), 'title_entity_first_position'].median()), 3),
    })
    features.to_csv(OUTPUT_DIR / partition / 'news_entity_prominence.csv', index=False, float_format='%.6g')
display(pd.DataFrame(summary_rows))
display(prominence_by_partition['train']['title_start_type_group'].value_counts().rename('train articles'))

,partition,articles,entity_mentions,mentions_without_offsets,title_start_strict_offset_0,title_start_with_leading_punctuation,title_entity_absent_%,abstract_entity_absent_%,median_title_first_position
0,train,43370,130664,488,14253,14364,27.05,27.36,0.083
1,validation,38421,113374,415,12495,12590,27.59,28.21,0.082
2,test,42416,126975,460,13997,14102,26.68,27.59,0.082


title_start_type_group
none            29006
person           4807
organisation     4453
other            2718
place            2386
Name: train articles, dtype: int64

### C2. Do prominent entities relate to clicks?

Two views, because they answer different questions:

1. **Pooled training CTR** by lead-entity group (descriptive, mixes different impressions).
2. **Within-impression ranking on validation**, which is what the recommender actually needs. A feature can show a large pooled CTR gap and still barely change the order inside one slate.

The lead type group is also turned into a number using smoothed **training** CTR per group (fitted on train only), then scored on validation.

In [14]:
train_rows = read_candidates('train').astype({'news_id': str}).merge(prominence_by_partition['train'], on='news_id', how='left', validate='many_to_one')
pooled = train_rows.groupby('title_start_type_group')['label'].agg(ctr='mean', exposures='size')
pooled['ctr_%'] = (100 * pooled.pop('ctr')).round(2)
display(pooled.sort_values('ctr_%', ascending=False))
group_stats = train_rows.groupby('title_start_type_group')['label'].agg(['sum', 'size'])
prior = train_rows['label'].mean()
type_rate = (group_stats['sum'] + 100 * prior) / (group_stats['size'] + 100)
del train_rows

val_rows = read_candidates('validation').astype({'news_id': str}).merge(prominence_by_partition['validation'], on='news_id', how='left', validate='many_to_one')
val_rows['event_key'] = val_rows['event_key'].astype(str)
val_rows['lead_type_train_ctr'] = val_rows['title_start_type_group'].map(type_rate).fillna(prior)
prominence_screen = univariate_table(val_rows, [
    'has_entity_at_title_start', 'lead_type_train_ctr', 'title_entity_first_position',
    'abstract_entity_first_position', 'title_entity_absent'],
    sign={'title_entity_first_position': -1, 'abstract_entity_first_position': -1, 'title_entity_absent': -1})
diagnostics['prominence_screen_validation'] = prominence_screen
display(prominence_screen)

,exposures,ctr_%
title_start_type_group,,
person,437133,5.99
other,243101,5.61
organisation,281026,5.52
none,2337363,3.60
place,90719,2.26


,feature,direction,impressions,AUC,MRR,nDCG@5,nDCG@10
0,has_entity_at_title_start,higher is better,61894,0.5078,0.2160,0.2209,0.2786
1,lead_type_train_ctr,higher is better,61894,0.5120,0.2184,0.2243,0.2817
2,title_entity_first_position,lower is better,61894,0.5127,0.2163,0.2220,0.2794
3,abstract_entity_first_position,lower is better,61894,0.5016,0.2123,0.2188,0.2751
4,title_entity_absent,lower is better,61894,0.5095,0.2125,0.2164,0.2746


**Reading C2.** Pooled over training exposures, titles led by a person (5.99% CTR), organisation (5.52%) or other entity (5.61%) are clicked far more than titles with no lead entity (3.60%), while titles led by a place are clicked less (2.26%). Inside an impression, though, these features move the ranking only slightly (AUC 0.508 for the indicator, 0.512 for the train-fitted lead-type rate). Much of the pooled gap comes from which slates contain such articles, not from users preferring them within a slate. Report both, and let the model use the lead type rather than the plain indicator, since the place effect has the opposite sign.

### C3. Test position-weighted entity aggregation

A model-free comparison isolates the aggregation choice. For each validation impression, the user profile is the mean entity vector of the supplied history articles, and each candidate is scored by cosine similarity to that profile. Only the weighting scheme changes between rows of the table; history, candidates and scoring stay fixed.

- **all**: every validation impression; candidates or users without any entity vector get score 0.
- **covered**: the same candidate rows for every scheme, namely those where the uniform scheme gives both a user profile and a candidate vector.

Two selections are fixed before running the grid, both by **covered nDCG@10** (the group's model-selection metric): the best scheme overall, and the best scheme that keeps abstract entities (abstract weight above 0), because title-only schemes leave many more articles without any entity vector. A paired per-impression difference with a 95% interval then checks each gain over uniform against impression-level noise. TransE vectors come from the `entity_embedding.vec` of the partition's own source (train file for train and validation, dev file for test); the two files agree exactly on shared entities.

In [15]:
SCHEMES = {
    'uniform': (1.0, 0.0),
    'abstract_half': (0.5, 0.0),
    'position_decay_1': (1.0, 1.0),
    'abstract_half_decay_1': (0.5, 1.0),
    'abstract_half_decay_2': (0.5, 2.0),
    'abstract_quarter_decay_2': (0.25, 2.0),
    'abstract_tenth_decay_1': (0.1, 1.0),
    'abstract_tenth_decay_2': (0.1, 2.0),
    'title_only': (0.0, 0.0),
    'title_only_decay_1': (0.0, 1.0),
    'title_only_decay_2': (0.0, 2.0),
}
vector_sources = {'train': 'MINDsmall_train', 'validation': 'MINDsmall_train', 'test': 'MINDsmall_dev'}
entity_tables = {source: load_entity_vectors(RAW_DIR / source / 'entity_embedding.vec') for source in set(vector_sources.values())}
shared = set(entity_tables['MINDsmall_train'][0]) & set(entity_tables['MINDsmall_dev'][0])
gap = max(float(np.abs(entity_tables['MINDsmall_train'][1][entity_tables['MINDsmall_train'][0][e]]
                       - entity_tables['MINDsmall_dev'][1][entity_tables['MINDsmall_dev'][0][e]]).max()) for e in shared)
print(f'Entity vectors: train file {len(entity_tables["MINDsmall_train"][0]):,}, dev file {len(entity_tables["MINDsmall_dev"][0]):,}, '
      f'shared {len(shared):,}, largest difference on shared entities {gap}')

events = read_events('validation', history=True)
candidates = read_candidates('validation')
news_ids = prominence_by_partition['validation']['news_id'].astype(str).tolist()
row_of = pd.Series(np.arange(len(news_ids)), index=news_ids)
event_of = pd.Series(np.arange(len(events)), index=events['event_key'].astype(str))
history = events[['event_key', 'history_ids']].explode('history_ids').dropna()
history_article = row_of.reindex(history['history_ids'].astype(str)).to_numpy()
history_event = event_of.reindex(history['event_key'].astype(str)).to_numpy()
candidate_article = row_of.reindex(candidates['news_id'].astype(str)).to_numpy()
candidate_event = event_of.reindex(candidates['event_key'].astype(str)).to_numpy()
index, matrix = entity_tables['MINDsmall_train']


def profile_scores(vectors, has_vector):
    profile = np.zeros((len(events), vectors.shape[1]), dtype='float32')
    counts = np.zeros(len(events))
    usable = has_vector[history_article]
    np.add.at(profile, history_event[usable], vectors[history_article[usable]])
    np.add.at(counts, history_event[usable], 1)
    profile /= np.maximum(counts, 1)[:, None]
    profile_norm = np.linalg.norm(profile, axis=1)
    vector_norm = np.linalg.norm(vectors, axis=1)
    scores = np.zeros(len(candidates))
    for start in range(0, len(candidates), 500_000):
        e = candidate_event[start:start + 500_000]
        a = candidate_article[start:start + 500_000]
        scores[start:start + 500_000] = (profile[e] * vectors[a]).sum(1) / np.maximum(profile_norm[e] * vector_norm[a], 1e-12)
    valid = (counts[candidate_event] > 0) & has_vector[candidate_article]
    return np.where(valid, scores, 0.0), valid


scored = pd.DataFrame({'event_key': candidates['event_key'].astype(str).to_numpy(), 'label': candidates['label'].to_numpy()})
rows, covered = [], None
for name, (abstract_weight, decay) in SCHEMES.items():
    weights = mention_weights(mentions_by_partition['validation'], abstract_weight=abstract_weight, decay=decay)
    vectors, missing = aggregate_entity_vectors(news_ids, mentions_by_partition['validation'], weights, index, matrix)
    scored[name], valid = profile_scores(vectors, missing == 0)
    if covered is None:
        covered = valid
    rows.append({'scheme': name, 'abstract_weight': abstract_weight, 'decay': decay,
                 'articles_with_vector_%': round(100 * float((missing == 0).mean()), 2),
                 **{f'all_{k}': v for k, v in impression_metrics(scored, name).items() if k != 'impressions'},
                 **{f'covered_{k}': v for k, v in impression_metrics(scored.loc[covered], name).items()}})
weighting = pd.DataFrame(rows)
diagnostics['entity_weighting_validation'] = weighting
display(weighting)

Entity vectors: train file 26,904, dev file 22,893, shared 18,346, largest difference on shared entities 0.0


,scheme,abstract_weight,decay,articles_with_vector_%,all_AUC,all_MRR,all_nDCG@5,all_nDCG@10,covered_impressions,covered_AUC,covered_MRR,covered_nDCG@5,covered_nDCG@10
0,uniform,1.00,0.0,85.41,0.5438,0.2338,0.2411,0.3013,51022,0.5519,0.2490,0.2589,0.3238
1,abstract_half,0.50,0.0,85.41,0.5457,0.2355,0.2435,0.3032,51022,0.5553,0.2512,0.2621,0.3264
2,position_decay_1,1.00,1.0,85.41,0.5464,0.2365,0.2444,0.3045,51022,0.5564,0.2525,0.2631,0.3280
3,abstract_half_decay_1,0.50,1.0,85.41,0.5469,0.2365,0.2448,0.3047,51022,0.5576,0.2528,0.2639,0.3283
4,abstract_half_decay_2,0.50,2.0,85.41,0.5476,0.2372,0.2460,0.3055,51022,0.5583,0.2535,0.2652,0.3293
5,abstract_quarter_decay_2,0.25,2.0,85.41,0.5482,0.2376,0.2466,0.3062,51022,0.5595,0.2541,0.2661,0.3303
6,abstract_tenth_decay_1,0.10,1.0,85.41,0.5497,0.2386,0.2483,0.3073,51022,0.5620,0.2554,0.2681,0.3316
7,abstract_tenth_decay_2,0.10,2.0,85.41,0.5493,0.2383,0.2479,0.3072,51022,0.5613,0.2551,0.2678,0.3315
8,title_only,0.00,0.0,71.85,0.5422,0.2414,0.2532,0.3101,51022,0.5531,0.2593,0.2747,0.3354
9,title_only_decay_1,0.00,1.0,71.85,0.5419,0.2415,0.2536,0.3101,51022,0.5525,0.2596,0.2752,0.3355


In [16]:
ranked = weighting.sort_values('covered_nDCG@10', ascending=False)
SELECTED_SCHEME = ranked.iloc[0]['scheme']
SELECTED_FULL_COVERAGE = ranked.loc[ranked['abstract_weight'].gt(0)].iloc[0]['scheme']
for label, name in [('Best overall', SELECTED_SCHEME), ('Best keeping abstract entities', SELECTED_FULL_COVERAGE)]:
    print(f'{label}: {name} (abstract weight {SCHEMES[name][0]}, decay {SCHEMES[name][1]})')
paired = pd.DataFrame([{'comparison': f'{name} minus uniform', **paired_difference(scored.loc[covered], 'uniform', name, metric)}
                       for name in dict.fromkeys([SELECTED_SCHEME, SELECTED_FULL_COVERAGE, 'abstract_half_decay_1'])
                       for metric in ('AUC', 'nDCG@10') if name != 'uniform'])
diagnostics['entity_weighting_paired'] = paired
display(paired)
del events, candidates, history, scored, val_rows
_ = gc.collect()

Best overall: title_only_decay_1 (abstract weight 0.0, decay 1.0)
Best keeping abstract entities: abstract_tenth_decay_1 (abstract weight 0.1, decay 1.0)


,comparison,metric,mean_difference,ci_low,ci_high,impressions
0,title_only_decay_1 minus uniform,AUC,0.0006,-0.0022,0.0034,51022
1,title_only_decay_1 minus uniform,nDCG@10,0.0117,0.0095,0.0138,51022
2,abstract_tenth_decay_1 minus uniform,AUC,0.0100,0.0086,0.0115,51022
3,abstract_tenth_decay_1 minus uniform,nDCG@10,0.0077,0.0063,0.0091,51022
4,abstract_half_decay_1 minus uniform,AUC,0.0057,0.0048,0.0066,51022
5,abstract_half_decay_1 minus uniform,nDCG@10,0.0045,0.0035,0.0055,51022


**Interpretation.** Weighting matters, but the effects are small in absolute terms because a cosine profile is a weak ranker on its own (about 0.55 AUC).

- Down-weighting abstract entities and later positions helps on both metrics. The prior hypothesis (abstract weight 0.5, decay 1) beats uniform by +0.0057 AUC [0.0048, 0.0066] and +0.0045 nDCG@10 [0.0035, 0.0055]. The best scheme that keeps abstract entities, `abstract_tenth_decay_1`, gains +0.0100 AUC [0.0086, 0.0115] and +0.0077 nDCG@10 [0.0063, 0.0091], with the same 85% article coverage as uniform.
- Position decay helps when abstract entities are included (uniform to `position_decay_1`: covered nDCG@10 0.3238 to 0.3280) but adds almost nothing inside titles alone, which are short.
- Title-only schemes give the best nDCG@10 (+0.0117) but no significant AUC gain, and 28% of articles lose their entity vector (14% for the other schemes).

**Recommendation for the advanced models.** Use `abstract_tenth_decay_1` as the default entity vector in the shared article encoder (significant on both metrics, full coverage), keep `uniform` for the ablation, and try `title_only_decay_1` as a variant. Keep `has_entity_at_title_start`, `title_start_type_group` and the first-position columns as article inputs. Confirm the choice inside the model with everything else held fixed, since a learned encoder can reweight entities itself.

In [17]:
exported = []
for partition in PARTITIONS:
    index, matrix = entity_tables[vector_sources[partition]]
    news_ids = prominence_by_partition[partition]['news_id'].astype(str).tolist()
    flags = pd.DataFrame({'news_id': news_ids})
    for name in dict.fromkeys(['uniform', SELECTED_FULL_COVERAGE, SELECTED_SCHEME]):
        abstract_weight, decay = SCHEMES[name]
        weights = mention_weights(mentions_by_partition[partition], abstract_weight=abstract_weight, decay=decay)
        vectors, missing = aggregate_entity_vectors(news_ids, mentions_by_partition[partition], weights, index, matrix)
        np.save(OUTPUT_DIR / partition / f'news_entity_vectors_{name}.npy', vectors)
        flags[f'entity_vector_missing_{name}'] = missing
        exported.append({'partition': partition, 'scheme': name, 'rows': vectors.shape[0], 'dim': vectors.shape[1],
                         'missing_%': round(100 * float(missing.mean()), 2)})
    flags.to_csv(OUTPUT_DIR / partition / 'news_entity_vector_index.csv', index=False)
display(pd.DataFrame(exported))

,partition,scheme,rows,dim,missing_%
0,train,uniform,43370,100,13.83
1,train,abstract_tenth_decay_1,43370,100,13.83
2,train,title_only_decay_1,43370,100,27.59
3,validation,uniform,38421,100,14.59
4,validation,abstract_tenth_decay_1,38421,100,14.59
5,validation,title_only_decay_1,38421,100,28.15
6,test,uniform,42416,100,13.89
7,test,abstract_tenth_decay_1,42416,100,13.89
8,test,title_only_decay_1,42416,100,27.23


## Export summary

Every partition folder contains the files below. Join candidate-level files on `(event_key, candidate_position)` and news-level files on `news_id` with `validate='many_to_one'`. Row *i* of each `.npy` matrix belongs to row *i* of `news_entity_vector_index.csv`.

In [18]:
for partition in PARTITIONS:
    encoded_by_partition[partition].to_csv(OUTPUT_DIR / partition / 'news_category_encoding.csv', index=False)
vocab['category'].to_csv(OUTPUT_DIR / 'category_vocab.csv', index=False)
vocab['subcategory'].to_csv(OUTPUT_DIR / 'subcategory_vocab.csv', index=False)
vocab['pairs'].to_csv(OUTPUT_DIR / 'subcategory_pairs.csv', index=False)
for name, table in diagnostics.items():
    table.to_csv(OUTPUT_DIR / f'diagnostic_{name}.csv', index=False)

dictionary = pd.DataFrame([
    ('category_id', 'news_category_encoding', 'News metadata', 'Training-sorted category code, 0 unknown', 'int32', 'Article', 'No'),
    ('subcategory_id', 'news_category_encoding', 'News metadata', '(category, subcategory) code; rare/unseen pairs use the category __RARE__ bucket; 0 unknown', 'int32', 'Article', 'No'),
    ('category_unknown, subcategory_backoff', 'news_category_encoding', 'News metadata', 'Flags for unknown category and non-individual subcategory code', 'int8', 'Article', 'No'),
    ('candidate_history_subcategory_share', 'candidate_subcategory_match', 'Behaviors + News', 'Share of supplied history in the candidate (category, subcategory) pair', 'float32', 'Candidate', 'No'),
    ('candidate_history_subcategory_count, history_subcategory_missing', 'candidate_subcategory_match', 'Behaviors + News', 'Matching history count; flag for no usable history subcategory', 'int', 'Candidate', 'No'),
    *[(column, 'candidate_history_{online,frozen}', 'Behaviors (labels of earlier impressions only)', 'See Part B table', 'float32' if 'ctr' in column or 'hours' in column else 'int', 'Candidate', 'Earlier impressions only')
      for column in HISTORY_COLUMNS],
    ('has_entity_at_title_start', 'news_entity_prominence', 'news.tsv entity offsets', 'Entity starts the title (leading punctuation allowed)', 'int8', 'Article', 'No'),
    ('title_start_entity_type, title_start_type_group', 'news_entity_prominence', 'news.tsv entity types', 'Type of the lead entity; person/organisation/place/other/none', 'string', 'Article', 'No'),
    ('title_entity_first_position, abstract_entity_first_position', 'news_entity_prominence', 'news.tsv entity offsets', 'Earliest offset / text length; 1.0 when absent', 'float32', 'Article', 'No'),
    ('title_entity_absent, abstract_entity_absent', 'news_entity_prominence', 'news.tsv entity offsets', 'No located entity in that section', 'int8', 'Article', 'No'),
    ('news_entity_vectors_<scheme>.npy', 'entity vectors', 'news.tsv + entity_embedding.vec', 'Weighted mean of 100-d TransE vectors; zero row when missing', 'float32[100]', 'Article', 'No'),
], columns=['feature', 'file', 'source', 'construction', 'data_type', 'level', 'uses_labels'])
dictionary.to_csv(OUTPUT_DIR / 'feature_dictionary.csv', index=False)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()


provenance = {
    'source_dataset_version': manifest['dataset_version'],
    'source_manifest_sha256': sha256(PROCESSED_DIR / 'manifest.json'),
    'partitions': PARTITIONS,
    'category_min_articles': MIN_ARTICLES,
    'history': history_meta,
    'history_modes': {'online': 'time < t from any partition', 'frozen': 'train: time < t; later partitions: time < partition start'},
    'entity_weight_formula': 'section_weight * exp(-decay * normalised_position); entity counted once with max weight',
    'entity_schemes_exported': list(dict.fromkeys(['uniform', SELECTED_FULL_COVERAGE, SELECTED_SCHEME])),
    'entity_scheme_selected_overall': SELECTED_SCHEME,
    'entity_scheme_selected_keeping_abstract': SELECTED_FULL_COVERAGE,
    'entity_scheme_selection_rule': 'highest covered nDCG@10 on validation (overall, and among abstract_weight > 0), fixed before the grid',
    'entity_scheme_parameters': {name: dict(zip(['abstract_weight', 'decay'], SCHEMES[name]))
                                 for name in dict.fromkeys(['uniform', SELECTED_FULL_COVERAGE, SELECTED_SCHEME])},
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(provenance, indent=2) + '\n')

written = sorted(p for p in OUTPUT_DIR.rglob('*') if p.is_file())
listing = pd.DataFrame({'file': [str(p.relative_to(OUTPUT_DIR)) for p in written],
                        'MB': [round(p.stat().st_size / 1e6, 1) for p in written]})
display(listing)
print(f'Total {listing["MB"].sum():.0f} MB written in {(time.time() - started) / 60:.1f} minutes')
display(dictionary)

,file,MB
0,category_vocab.csv,0.0
1,diagnostic_entity_weighting_paired.csv,0.0
2,diagnostic_entity_weighting_validation.csv,0.0
3,diagnostic_history_screen_validation.csv,0.0
4,diagnostic_prominence_screen_validation.csv,0.0
5,diagnostic_subcategory_match_validation.csv,0.0
6,feature_dictionary.csv,0.0
7,manifest.json,0.0
8,subcategory_pairs.csv,0.0
9,subcategory_vocab.csv,0.0


Total 555 MB written in 13.8 minutes


,feature,file,source,construction,data_type,level,uses_labels
0,category_id,news_category_encoding,News metadata,"Training-sorted category code, 0 unknown",int32,Article,No
1,subcategory_id,news_category_encoding,News metadata,"(category, subcategory) code; rare/unseen pair...",int32,Article,No
2,"category_unknown, subcategory_backoff",news_category_encoding,News metadata,Flags for unknown category and non-individual ...,int8,Article,No
3,candidate_history_subcategory_share,candidate_subcategory_match,Behaviors + News,Share of supplied history in the candidate (ca...,float32,Candidate,No
4,"candidate_history_subcategory_count, history_s...",candidate_subcategory_match,Behaviors + News,Matching history count; flag for no usable his...,int,Candidate,No
5,art_prior_exposures,"candidate_history_{online,frozen}",Behaviors (labels of earlier impressions only),See Part B table,int,Candidate,Earlier impressions only
6,art_prior_clicks,"candidate_history_{online,frozen}",Behaviors (labels of earlier impressions only),See Part B table,int,Candidate,Earlier impressions only
7,art_prior_ctr,"candidate_history_{online,frozen}",Behaviors (labels of earlier impressions only),See Part B table,float32,Candidate,Earlier impressions only
8,art_exposures_1h,"candidate_history_{online,frozen}",Behaviors (labels of earlier impressions only),See Part B table,int,Candidate,Earlier impressions only
9,art_clicks_1h,"candidate_history_{online,frozen}",Behaviors (labels of earlier impressions only),See Part B table,int,Candidate,Earlier impressions only


## Using these features in the models

```python
OUT = Path(os.environ['BT4222_CHRIS_FEATURES_DIR'])
partition = 'validation'
history = pd.read_csv(OUT / partition / 'candidate_history_online.csv.gz', dtype={'event_key': 'string'})
match = pd.read_csv(OUT / partition / 'candidate_subcategory_match.csv.gz', dtype={'event_key': 'string'})
cats = pd.read_csv(OUT / partition / 'news_category_encoding.csv', dtype={'news_id': 'string'})
prominence = pd.read_csv(OUT / partition / 'news_entity_prominence.csv', dtype={'news_id': 'string'})

examples = (candidates
            .merge(history, on=['event_key', 'candidate_position'], how='left', validate='one_to_one')
            .merge(match, on=['event_key', 'candidate_position'], how='left', validate='one_to_one')
            .merge(cats, on='news_id', how='left', validate='many_to_one')
            .merge(prominence, on='news_id', how='left', validate='many_to_one'))

index = pd.read_csv(OUT / partition / 'news_entity_vector_index.csv', dtype={'news_id': 'string'})
vectors = np.load(OUT / partition / 'news_entity_vectors_<selected scheme>.npy')
row_of = pd.Series(np.arange(len(index)), index=index['news_id'])
```

Notes for modelling:

- Apply `log1p` to count columns and fit any scaler on training rows only.
- `category_id` and `subcategory_id` are for embedding layers; size them from `category_vocab.csv` and `subcategory_vocab.csv`.
- Swap `candidate_history_online` for `candidate_history_frozen` to run the online vs frozen ablation without changing anything else.
- Compute features on the full candidate table before `src.sampling.sample_negatives`, then join by `(event_key, candidate_position)`.